# JakCCTV — Dasbor Monitoring Busway di Kaggle (GPU)

Menjalankan dasbor (live 4 kamera + log + peta + insight) di GPU Kaggle, lalu membuat **link publik** lewat Cloudflare Tunnel supaya teman-teman bisa membuka dari browser masing-masing.

**Cara termudah (sumber GitHub, default):** cukup Settings: **Accelerator GPU T4**, **Internet On**, lalu Run All. Tidak perlu dataset. Update kode = push ke GitHub, lalu jalankan ulang notebook.

**Alternatif (sumber dataset, ubah `SUMBER = 'dataset'` di sel 1):**
1. Di laptop: `python src/siapkan_dasbor_cloud.py` → `outputs/dasbor_cloud.zip`.
2. Kaggle → Create → New Dataset → upload `dasbor_cloud.zip`, judul `jakcctv-dasbor`.
3. Notebook ini: Add Input → dataset `jakcctv-dasbor`. Settings: **Accelerator GPU T4**, **Internet On**.

**Menjalankan:** jalankan sel 1–5 berurutan (boleh Run All). Link muncul di sel 5:
- **Link penonton** → bagikan ke teman (hanya bisa melihat).
- **Link admin** → untuk kamu (bisa on/off kamera, verifikasi ✓/✗, upload video). Jangan dibagikan.

**Catatan penting:**
- Biarkan tab notebook ini tetap terbuka. Sesi interaktif Kaggle berhenti kalau lama tidak aktif, dan sesi maksimal ±12 jam. Kalau sesi berhenti, link ikut mati.
- Link `trycloudflare.com` berubah setiap kali dijalankan ulang.
- Log/snapshot tersimpan di `/kaggle/working/app/outputs/monitor`. Unduh lewat sel 7 sebelum sesi berakhir kalau mau disimpan.

In [ ]:
# 1) Ambil aplikasi ke /kaggle/working/app
#    SUMBER = 'github'  -> kode terbaru langsung dari GitHub (update cukup push ke repo, tanpa upload zip)
#    SUMBER = 'dataset' -> dari input dataset jakcctv-dasbor (dasbor_cloud.zip)
SUMBER = 'github'
REPO = 'https://github.com/Paradiess-svg/jakcctv-busway-monitor.git'
import os, shutil, glob, subprocess, sys, time, secrets, re, json, urllib.request
APP = '/kaggle/working/app'
if os.path.exists(APP):
    shutil.rmtree(APP)
if SUMBER == 'github':
    subprocess.run(['git', 'clone', '--depth', '1', REPO, APP], check=True)
else:
    kandidat = glob.glob('/kaggle/input/**/jakcctv_app/dashboard/server.py', recursive=True)
    if not kandidat:   # zip belum diekstrak otomatis
        z = glob.glob('/kaggle/input/**/dasbor_cloud.zip', recursive=True)
        assert z, 'Dataset jakcctv-dasbor belum ditambahkan (Add Input), atau pakai SUMBER = github'
        shutil.unpack_archive(z[0], '/kaggle/working/_ekstrak')
        kandidat = glob.glob('/kaggle/working/_ekstrak/**/jakcctv_app/dashboard/server.py', recursive=True)
    assert len(kandidat) == 1, kandidat
    shutil.copytree(os.path.dirname(os.path.dirname(kandidat[0])), APP)
print('Aplikasi di', APP, f'(sumber: {SUMBER})'); print(sorted(os.listdir(APP)))

In [ ]:
# 2) Pasang library (torch & ffmpeg biasanya sudah ada di Kaggle)
!pip install -q ultralytics fastapi "uvicorn[standard]" python-multipart lap
!which ffmpeg || (apt-get -qq update && apt-get -qq install -y ffmpeg)
import torch; print('GPU:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
# 3) Tes: apakah server Kaggle bisa membuka stream JakCCTV? (server Kaggle ada di luar negeri)
import yaml
cfg = yaml.safe_load(open(f'{APP}/config/config.yaml'))
kam = cfg['dasbor']['kamera_default']
sys.path.insert(0, f'{APP}/src')
from capture import FFMPEG_INPUT, configure_hls_opts   # opsi HLS sesuai versi ffmpeg (sama dengan server)
configure_hls_opts('ffmpeg')
for k in kam:
    url = cfg['cameras'][k]['url']
    r = subprocess.run(['ffmpeg', *FFMPEG_INPUT, '-i', url,
                        '-frames:v', '1', '-f', 'null', '-'], capture_output=True, text=True, timeout=60)
    print(f"{k:22s}", 'OK' if r.returncode == 0 else 'GAGAL: ' + r.stderr.strip()[-200:])
# Kalau semua GAGAL: stream JakCCTV kemungkinan menolak akses dari luar Indonesia -> pakai mode laptop.

In [ ]:
# 4) Jalankan server dasbor di latar belakang (token admin dibuat acak)
TOKEN = secrets.token_urlsafe(8)
env = dict(os.environ, TJ_TOKEN=TOKEN, PYTHONUNBUFFERED='1')
log_srv = open('/kaggle/working/server.log', 'w')
srv = subprocess.Popen([sys.executable, 'dashboard/server.py', '--port', '8000'], cwd=APP, env=env,
                       stdout=log_srv, stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        print(urllib.request.urlopen('http://127.0.0.1:8000/api/info', timeout=2).read().decode()); break
    except Exception:
        time.sleep(2)
else:
    print(open('/kaggle/working/server.log').read()[-3000:])

In [ ]:
# 5) Link publik lewat Cloudflare Tunnel (gratis, tanpa akun)
if not os.path.exists('/kaggle/working/cloudflared'):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64',
                               '/kaggle/working/cloudflared')
    os.chmod('/kaggle/working/cloudflared', 0o755)
log_cf = open('/kaggle/working/cloudflared.log', 'w')
cf = subprocess.Popen(['/kaggle/working/cloudflared', 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8000'],
                      stdout=log_cf, stderr=subprocess.STDOUT)
LINK = None
for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/kaggle/working/cloudflared.log').read())
    if m:
        LINK = m.group(0); break
    time.sleep(2)
print('LINK PENONTON :', LINK)
print('LINK ADMIN    :', f'{LINK}/?admin={TOKEN}   <- jangan dibagikan')
# Live otomatis menyala saat link pertama kali dibuka.

In [ ]:
# 6) Pantau (opsional): status kamera tiap 60 detik. Hentikan sel ini kapan saja (server tetap jalan).
try:
    while True:
        s = json.loads(urllib.request.urlopen('http://127.0.0.1:8000/api/live/status').read())
        print(time.strftime('%H:%M:%S'), ' | '.join(f"{k['kamera']}: {k['stream']} {k.get('fps_analisis', 0)} fps TJ {k['hitung']['tj']} non {k['hitung']['non_tj']}" for k in s['kamera']))
        time.sleep(60)
except KeyboardInterrupt:
    pass

In [ ]:
# 7) Simpan log & snapshot (zip) -> unduh dari tab Output, lalu matikan semuanya
shutil.make_archive('/kaggle/working/hasil_dasbor', 'zip', f'{APP}/outputs/monitor')
print('Unduh /kaggle/working/hasil_dasbor.zip dari tab Output')
try:
    urllib.request.urlopen(urllib.request.Request('http://127.0.0.1:8000/api/live/stop', method='POST', headers={'X-Token': TOKEN}))
except Exception as e:
    print(e)
for p in (cf, srv):
    p.terminate()
print('Server & tunnel dimatikan')